# LogVar2FJ 2 — what the calibration identifies, measured

A ladder of vanillas does not close this model. Two directions are flat in it: the residual's
tail `Alpha`, and the split of the spot skew between the residual's skew `Beta` and the leverage
product `Rho_S x Sigma_S`. On every book ladder those are held by priors, and notebook 1's report
says so twice — `Alpha` lands on its class prior, and its prior row prints at several times one
quote row.

This notebook measures what would have to be quoted for `Alpha` to be fitted instead, and shows
that the windows a desk would ask for by default do not do it.

**The measurement is self-consistent by construction.** Take one written factor, re-quote the
five-expiry ladder at *that factor's own* vanilla vols, and the factor is exactly the vanilla
objective's minimum — every quote reprices to the digit. What is left to identify is then
precisely what the vanillas are flat in, and the targets are that same factor's own forward
smiles, read back through the family's own reader. Nothing is approximated and nothing is fitted
to noise.

The construction and its helpers live in `tests/test_logvar2fj_json.py`; this notebook imports
that module and calls them, so the notebook and the repository's own gate run the same documents.

In [1]:
import copy, io, json, logging, os, sys, time

HERE = os.path.abspath(os.getcwd())
REPO = HERE if os.path.isdir(os.path.join(HERE, 'derivus')) else os.path.dirname(HERE)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'tests'))

import torch
import derivus as rf
from derivus import bootstrappers
import test_logvar2fj_json as T

print('derivus   ', os.path.dirname(rf.__file__))
print('device    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('world     ', os.path.relpath(T.WORLD, REPO))
print('factor    ', T.FACTOR)

derivus    C:\Users\Vretiel\PycharmProjects\derivus\.claude\worktrees\agent-aec710af441821806\derivus
device     NVIDIA GeForce RTX 3090
world      tests\fixtures\data\logvar2fj_world.json
factor     LogVar2FJModelParameters.INDEX_A


In [2]:
def fit(sections, name):
    """`(the written factor, the report the fit printed)` - the same document `T._fit` builds,
    with every root handler replaced so the report is this notebook's to render."""
    buf, root = io.StringIO(), logging.getLogger()
    saved, level = root.handlers[:], root.level
    root.handlers, root.level = [logging.StreamHandler(buf)], logging.INFO
    try:
        cx = rf.Context()
        cx.load_json((T._dumps(T._job(T._base(), (), **sections)), name + '.json'))
        cx.bootstrap()
    finally:
        root.handlers, root.level = saved, level
    return cx.current_cfg.params['Price Factors'].get(T.FACTOR), buf.getvalue()


class quiet:
    """Every root handler replaced for the block - the reader's INFO and WARNING lines are the
    family's own and belong in a report, not in a cell."""

    def __enter__(self):
        root = logging.getLogger()
        self.saved, self.level = root.handlers[:], root.level
        root.handlers, root.level = [logging.StreamHandler(io.StringIO())], logging.INFO

    def __exit__(self, *failure):
        root = logging.getLogger()
        root.handlers, root.level = self.saved, self.level


def forward_rows(source, tenors):
    """`WORLD_FACTOR`'s own forward smiles at `tenors`, as `Forward_Smiles` rows.

    The family's own reader at a WRITTEN factor - no fit runs - which is the seat a desk model's
    rows take later: the same table and the same fields, a vol per window and strike."""
    with quiet():
        cx = rf.Context()
        cx.load_json((T._dumps(T._job(T._base(), (), **dict(
            T._model_ladder(Forward_Tenors=tenors, Forward_Smile_Source=source),
            **{'Price Factors': {T.FACTOR: T.WORLD_FACTOR}}))), 'read.json'))
        params = cx.current_cfg.params
        family = bootstrappers.construct_bootstrapper(
            'LogVar2FJModelParameters',
            params['Bootstrapper Configuration']['LogVar2FJModelParameters'])
        smile = family.forward_smiles(
            params['System Parameters'], params['Price Models'], params['Price Factors'],
            params['Price Factor Interpolation'],
            bootstrappers.market_prices_for('LogVar2FJModelParameters',
                                            params['Market Prices']))[T.BLOCK]
    return [{'T1': T1, 'Delta': tenor, 'Strike': k, 'Target_Vol': smile[(T1, tenor)][k]}
            for T1, tenor in sorted(smile) for k in sorted(smile[(T1, tenor)])]


def lever(factor, name):
    return float(factor[name].array[0][1])

## The world factor

A fit of the world's ladder whose residual pair was moved away from the shipped class priors —
so the tail it carries is not the prior's, and a fit that lands on the prior has plainly not read
it off the data.

In [3]:
for name, value in sorted(T.WORLD_FACTOR.items()):
    if isinstance(value, dict):
        rows = value['.Curve']['data']
        print('%-12s %s' % (name, ', '.join('%g -> %.10g' % (t, v) for t, v in rows)))
    else:
        print('%-12s %s' % (name, value))
print()
print('the class prior on Alpha is %.1f; this world carries %.4f' % (
    T.ALPHA_PRIOR, T.WORLD_FACTOR['Alpha']['.Curve']['data'][0][1]))

Alpha        0 -> 20.91669436
Beta         0 -> -2.030895411
C_Min        0.12
Cap_A        3.8329158963830388
Kappa_L      0.5
Kappa_S      6.0
Residual_Law NIG
Rho_L        -0.4
Rho_S        0 -> -0.7939878539
Sigma_L      1.0
Sigma_S      0 -> 2.05363722
Xi_Curve     0 -> 0.04318756981, 0.0767123 -> 0.0470293452, 0.249315 -> 0.05089542481, 0.49863 -> 0.05447323288, 0.747945 -> 0.05807287748

the class prior on Alpha is 44.0; this world carries 20.9167


## The ladder, re-quoted at the factor's own vols

Fifteen quotes, three strikes at each of five expiries, at the vols `WORLD_FACTOR` itself prices
them to. The factor is now the vanilla objective's minimum exactly.

In [4]:
sections = T._model_ladder()
rows = sections['Market Prices'][T.BLOCK]['instrument']['European_Options']
print('%-12s %10s %6s %10s' % ('expiry', 'strike', 'type', 'vol'))
for row in rows:
    print('%-12s %10.4f %6s %10.6f' % (row['Expiry_Date']['.Timestamp'], row['Strike'],
                                       row['Option_Type'], row['Quoted_Market_Value']))

expiry           strike   type        vol
2024-07-26      90.1036    Put   0.249168
2024-07-26     100.1151   Call   0.200767
2024-07-26     110.1266   Call   0.171815
2024-09-27      90.3372    Put   0.239558
2024-09-27     100.3747   Call   0.202493
2024-09-27     110.4121   Call   0.167899
2024-12-27      90.6757    Put   0.233923
2024-12-27     100.7507   Call   0.204986
2024-12-27     110.8258   Call   0.177806
2025-03-28      91.0154    Put   0.231700
2025-03-28     101.1282   Call   0.207479
2025-03-28     111.2411   Call   0.185185
2025-06-27      91.3564    Put   0.231111
2025-06-27     101.5071   Call   0.209973
2025-06-27     111.6578   Call   0.190859


## The forward rows

Nine forward-start rows: a **one-month** window opening three months, six months and a year
ahead, three strikes each, at the world factor's own forward smiles. These are read back from the
written factor by the family's own reader — no fit runs — which is the seat a traded
forward-start mark or a desk model's smile takes in production, through the same fields.

Two sources price the same table as two different instruments. `Quotes` is the traded
forward-start, priced under the share measure. `Reference` is the ratio expectation
`E[(S_T2/S_T1 - k)+]` under the pricing measure, which is what a reference model reports. The
difference between them is the share-measure factor and nothing else.

In [5]:
started = time.time()
reference_rows = forward_rows('Reference', T.WORLD_TENORS)
quotes_rows = forward_rows('Quotes', T.WORLD_TENORS)
print('%s -> %d rows each, read in %.1f s\n' % (T.WORLD_TENORS, len(reference_rows),
                                                time.time() - started))
print('%10s %10s %8s %12s %12s' % ('T1, years', 'window', 'strike', 'Reference', 'Quotes'))
for a, b in zip(reference_rows, quotes_rows):
    print('%10.4f %10.4f %8.4f %12.6f %12.6f' % (float(a['T1']), float(a['Delta']),
                                                 a['Strike'], a['Target_Vol'], b['Target_Vol']))

3m:1m,6m:1m,1y:1m -> 9 rows each, read in 2.5 s

 T1, years     window   strike    Reference       Quotes
    0.2500     0.0833   0.9000     0.279972     0.274553
    0.2500     0.0833   1.0000     0.214578     0.208149
    0.2500     0.0833   1.1000     0.210342     0.203877
    0.5000     0.0833   0.9000     0.287960     0.280001
    0.5000     0.0833   1.0000     0.215047     0.206063
    0.5000     0.0833   1.1000     0.216237     0.207771
    1.0000     0.0833   0.9000     0.284984     0.279712
    1.0000     0.0833   1.0000     0.212310     0.202495
    1.0000     0.0833   1.1000     0.222403     0.212925


## Four fits of the same ladder

The block off; the block on under each source at a one-month window; and the block on at the
**declared default windows** — six months into six, a year into a year, a year into three months
— which is what a desk asked for a forward block would ask for first. Every fit is cold started,
so each one begins at the class priors.

In [6]:
DEFAULT_TENORS = '6m:6m,1y:1y,1y:3m'
default_rows = forward_rows('Reference', DEFAULT_TENORS)
print('%s -> %d rows' % (DEFAULT_TENORS, len(default_rows)))

plan = [('block off', dict(Max_Iterations=60, Forward_Tenors=T.WORLD_TENORS)),
        ('Reference, 1m', dict(Max_Iterations=60, Forward_Tenors=T.WORLD_TENORS,
                               Forward_Smile_Source='Reference', Forward_Smiles=reference_rows)),
        ('Quotes, 1m', dict(Max_Iterations=60, Forward_Tenors=T.WORLD_TENORS,
                            Forward_Smile_Source='Quotes', Forward_Smiles=quotes_rows)),
        ('Reference, default windows',
         dict(Max_Iterations=60, Forward_Tenors=DEFAULT_TENORS,
              Forward_Smile_Source='Reference', Forward_Smiles=default_rows))]

fits = {}
for tag, declared in plan:
    started = time.time()
    fits[tag] = fit(T._model_ladder(**declared), tag.split(',')[0].replace(' ', '_'))
    print('%-28s %6.1f s' % (tag, time.time() - started))

6m:6m,1y:1y,1y:3m -> 9 rows


block off                       9.0 s


Reference, 1m                  37.6 s


Quotes, 1m                     29.7 s


Reference, default windows     30.4 s


### What the four fits landed

`Alpha` and `Beta` are the residual; `Rho_S` and `Sigma_S` are the leverage. Beside each number
is the prior row the joint polish reported for that coordinate, as a multiple of one quote row at
the data's own RMS — a coordinate whose prior row is several quote rows is being held.

In [7]:
NAMES = ('Alpha', 'Beta', 'Rho_S', 'Sigma_S')
world = {n: T.WORLD_FACTOR[n]['.Curve']['data'][0][1] for n in NAMES}
print('%-28s %s' % ('', ''.join('%22s' % n for n in NAMES)))
print('%-28s %s' % ('the world', ''.join('%22.4f' % world[n] for n in NAMES)))
print('-' * 116)
for tag, (factor, report) in fits.items():
    ratios = T._prior_ratios(report)['6 joint polish']
    cells = []
    for n in NAMES:
        key = n if n in ratios else n + '[0y]'
        cells.append('%13.4f %7s' % (lever(factor, n),
                                     '(%.2fx)' % ratios[key] if key in ratios else '(-)'))
    print('%-28s %s' % (tag, ''.join(cells)))

                                              Alpha                  Beta                 Rho_S               Sigma_S
the world                                   20.9167               -2.0309               -0.7940                2.0536
--------------------------------------------------------------------------------------------------------------------
block off                          43.7623 (6.74x)     -13.4998 (4.34x)      -0.6972 (5.77x)       2.3720 (3.39x)
Reference, 1m                      20.2740 (2.35x)      -4.7779 (2.58x)      -0.6626 (4.30x)       2.3868 (4.57x)
Quotes, 1m                         28.0070 (3.08x)      -6.3069 (3.19x)      -0.6908 (5.30x)       2.3604 (4.60x)
Reference, default windows         43.4816 (7.55x)     -11.7014 (4.64x)      -0.6569 (7.47x)       2.6438 (5.01x)


In [8]:
def polish(report, vanillas_only=False):
    """The joint polish's own identification line - the one taken WITH the forward rows, or the
    second one taken at the same fitted parameters without them."""
    marker = 'identification, 6 joint polish:'
    if vanillas_only:
        marker = 'identification, 6 joint polish, vanillas only:'
    lines = [ln for ln in report.splitlines() if marker in ln]
    if not lines:
        return None, None
    values, norms = lines[-1].split('singular values')[1].split('; column norms')
    return (min(float(x) for x in values.split()),
            float(norms.split('Alpha[0y]')[1].split(',')[0]))


print('%-28s %14s %16s %16s' % ('', 'RMSE, vol pts', 'smallest s.v.', 'Alpha column norm'))
for tag, (factor, report) in fits.items():
    rmse = T._report_floats(report, 'RMSE', 'vol points unweighted')[0]
    smallest, alpha = polish(report)
    print('%-28s %14.3f %16.4f %16.2e' % (tag, rmse, smallest, alpha))

                              RMSE, vol pts    smallest s.v. Alpha column norm
block off                             0.337           0.1573         6.78e-05
Reference, 1m                         0.310           0.1997         4.33e-04
Quotes, 1m                            0.180           0.2282         2.40e-04
Reference, default windows            0.324           0.2523         6.30e-05


### The reading

`Alpha` is the whole of it. With the block off the fit lands on the class prior — it is not a
fitting failure, because the ladder is the model's own and reprices to the digit; there is
nothing in a vanilla to read the residual's tail off. Nine one-month rows recover it. The same
nine rows at the default windows recover nothing at all: over a quarter the residual's increment
has already aggregated to nearly Gaussian, so what those windows tilt by is the leverage, and a
block asked to identify the residual pair is asked for a **short** window.

`Beta` moves a long way and does not arrive. What the rows carry is the share of the skew the
forward smile sees, and at this ladder's resolution that is a direction rather than a digit.
The two sources agree on the direction and on the size of the move, and not on the digit: the
share measure is a reweighting of the same paths, and is worth about a vol point of target on
these windows.

In [9]:
off, one_month, default = fits['block off'][0], fits['Reference, 1m'][0], fits['Reference, default windows'][0]
print('Alpha: prior %.2f, world %.4f' % (T.ALPHA_PRIOR, world['Alpha']))
for tag in ('block off', 'Reference, 1m', 'Quotes, 1m', 'Reference, default windows'):
    value = lever(fits[tag][0], 'Alpha')
    print('  %-28s %8.4f  %+7.1f%% of the world, %+7.1f%% of the prior' % (
        tag, value, 100 * (value / world['Alpha'] - 1), 100 * (value / T.ALPHA_PRIOR - 1)))
print()
print('Beta: world %.4f' % world['Beta'])
for tag in ('block off', 'Reference, 1m', 'Quotes, 1m', 'Reference, default windows'):
    value = lever(fits[tag][0], 'Beta')
    print('  %-28s %8.4f  %.2fx the distance the block-off fit stands at' % (
        tag, value, abs(value - world['Beta']) / abs(lever(off, 'Beta') - world['Beta'])))

Alpha: prior 44.00, world 20.9167
  block off                     43.7623   +109.2% of the world,    -0.5% of the prior
  Reference, 1m                 20.2740     -3.1% of the world,   -53.9% of the prior
  Quotes, 1m                    28.0070    +33.9% of the world,   -36.3% of the prior
  Reference, default windows    43.4816   +107.9% of the world,    -1.2% of the prior

Beta: world -2.0309
  block off                    -13.4998  1.00x the distance the block-off fit stands at
  Reference, 1m                 -4.7779  0.24x the distance the block-off fit stands at
  Quotes, 1m                    -6.3069  0.37x the distance the block-off fit stands at
  Reference, default windows   -11.7014  0.84x the distance the block-off fit stands at


The joint polish also prints its identification table a second time at the same fitted parameters
**without** the forward rows, so the two readings differ in the rows and in nothing else.

In [10]:
report = fits['Reference, 1m'][1]
with_rows, without_rows = polish(report), polish(report, vanillas_only=True)
print('%-28s %16s %16s' % ('Reference, 1m', 'smallest s.v.', 'Alpha column norm'))
print('%-28s %16.4f %16.2e' % ('  with the forward rows', with_rows[0], with_rows[1]))
print('%-28s %16.4f %16.2e' % ('  the same theta*, without', without_rows[0], without_rows[1]))
print()
for tag in ('block off', 'Reference, 1m'):
    lines = fits[tag][1].splitlines()
    at = [n for n, ln in enumerate(lines) if 'identification, 6 joint polish:' in ln][-1]
    print(tag)
    print('   ' + lines[at + 1].strip())

Reference, 1m                   smallest s.v. Alpha column norm
  with the forward rows                0.1997         4.33e-04
  the same theta*, without             0.0897         2.72e-04

block off
   the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_S[0y] 5.77x, Beta[0y] 4.34x, Sigma_S[0y] 3.39x, Alpha[0y] 6.74x
Reference, 1m
   the PRIOR rows there, each as a multiple of ONE quote row at the data's own RMS: Rho_S[0y] 4.3x, Beta[0y] 2.58x, Sigma_S[0y] 4.57x, Alpha[0y] 2.35x


### What it cost the vanilla fit

A desk decides on this: what the forward rows did to the fit of the quotes that were actually
traded. On a world the model owns the rows point at the parameters that priced those quotes, so
the block pays for itself. A market source on a ladder the model cannot fit is where this reading
becomes a cost, and it is read in the same line.

In [11]:
for tag, (factor, report) in fits.items():
    print('%-28s %s' % (tag, [ln.strip() for ln in report.splitlines()
                              if 'vol points unweighted' in ln][-1]))

block off                    RMSE 0.337 vol points unweighted over 15 quotes, 0.337 vega-weighted (the objective's own); the bootstrap's ATM misses +7.1e-11, -5.1e-15, +3.2e-13, +6.5e-12, +3.4e-11
Reference, 1m                RMSE 0.310 vol points unweighted over 15 quotes, 0.310 vega-weighted (the objective's own); the bootstrap's ATM misses +8.2e-11, -2.2e-16, +3.0e-13, +6.1e-12, +3.2e-11
Quotes, 1m                   RMSE 0.180 vol points unweighted over 15 quotes, 0.180 vega-weighted (the objective's own); the bootstrap's ATM misses +7.7e-11, -4.0e-15, +3.1e-13, +6.4e-12, +3.3e-11
Reference, default windows   RMSE 0.324 vol points unweighted over 15 quotes, 0.324 vega-weighted (the objective's own); the bootstrap's ATM misses +7.5e-11, -4.4e-16, +3.4e-13, +6.7e-12, +3.4e-11


## What stays prior-held, and where production rows come from

**The leverage split stays prior-held.** `Rho_S` and `Sigma_S` move barely at all across the four
fits, and their prior rows stay at several quote rows in every one. The forward block conditions
the flat direction the split lives in; it does not resolve it. A desk that wants the split
measured has to declare it — a leverage pair with its own standard error — and the report will
then say the number is declared rather than fitted.

**The forward rows come from outside the engine.** There is no second dynamics model here to
serve as a reference, and a deterministic-vol model's forward smile is flat, so the rows that
would identify the tail in production are traded forward-start or cliquet marks under
`Forward_Smile_Source: Quotes`, or a desk model's forward smiles under `Reference` — the same
table and the same fields this notebook filled synthetically. Until either is supplied, every
production fit is vanilla-only with its priors stated in its own report, and the unquoted forward
skew is carried by the reserve of notebook 4 rather than fitted.

**And the window is a design choice, not a convenience.** A block asked for six months into six
months will report a clean fit and identify nothing. The rows have to be short.